# Implementing LSH on the N-most cited dataset

In [2]:
import json
from typing import Dict, Any
import numpy as np

In [3]:
from lsh import preprocess_lsh, lsh 
from lsh_utils.signatures import signatures

## Preprocess 

Keep ony the important part of the data

In [3]:
json_path_Nmost = '../data/processed/filtered_articles_Nmostcited.json'
data_Nmost = preprocess_lsh(dataset_path = json_path_Nmost)
print("preprocessing done")

Data succesfully loaded
preprocessing done


Save the preprocessed data into a json file

In [6]:
output_json_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Nmost, f, indent=4)

## Compute the signatures

Load the simplified data previously saved

In [12]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

Compute the signature matrix

In [13]:
q = 7 
b = 4
r = 5

signature_matrix_Nmost, idx_to_id_Nmost = signatures(
    doc_list=data_Nmost,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures: 100%|██████████| 24290/24290 [13:06<00:00, 30.90it/s]   

min hashing of the documents complete


Save the results

In [ ]:
np.save(file = f"../data/processed/signatures_lsh/Nmost_q{q}_b{b}_r{r}", arr = signature_matrix_Nmost)
output_json_path = f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Nmost, f, indent=4)

## Perform the research of the most relevant documents using LSH

load data and the saved signature matrix

In [4]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

q = 7 
b = 4
r = 5

try :
    signature_matrix_Nmost = np.load(f"../data/processed/signatures_lsh/Nmost_q{q}_b{b}_r{r}.npy")
    with open(f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id_Nmost: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

In [5]:
print(signature_matrix_Nmost.shape)

(20, 24290)


In [6]:
print(idx_to_id_Nmost[2])

{'id': '0911.0802', 'abstract': 'we construct analytic extensions of the pomeranskysenkov metrics with\nmultiple killing horizons and asymptotic regions we show that in our\nextensions the singularities associated to an obstruction to differentiability\nof the metric lie beyond event horizons we analyze the topology of the\nnonempty singular set which turns out to be parameterdependent we present\nnumerical evidence for stable causality of the domain of outer communications\nthe resulting global structure is somewhat reminiscent of that of kerr\nspacetime'}


Perform LSH to obtain the most similar documents to input (try with abstract of the first document as input)

In [ ]:
Most_similar, Scores = lsh(
        input = data_Nmost[0]['abstract'],
        signature_matrix = signature_matrix_Nmost,
        idx_to_id = idx_to_id_Nmost,
        m = signature_matrix_Nmost.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
        )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 46.34it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands:  25%|██▌       | 1/4 [00:00<00:00,  3.12it/s]

4.1169205434335116e-05 % of signatures computed ... 



...

LSH Bands:  75%|███████▌  | 3/4 [00:00<00:00,  3.61it/s]

2.0000411692054345 % of signatures computed ... 



LSH Bands: 100%|██████████| 4/4 [00:01<00:00,  3.73it/s]


3.0000411692054345 % of signatures computed ... 

LSH successfully performed to find similar candidates
Calculation of the actual similarities ...


Calculating Similarities: 100%|██████████| 45/45 [00:00<00:00, 75831.13it/s]


In [10]:
print("Most similar documents : " , Most_similar, '\n')
print("Scores : " , Scores)

Most similar documents :  ['1002.3982', '0802.1718', '1006.0106', '0806.3825', '1010.5141', '1110.4738', '1203.1672', '0807.0646', '0808.1161', '0904.0380', '0808.3413', '0811.4571', '1110.6838', '1103.0885', '1111.5608', '0802.4221', '0904.3198', '1005.3266', '0901.4348', '0903.3108', '0906.1523', '0903.2733', '0907.5115', '0706.0322', '0810.3296', '1005.4655', '0906.4370', '1003.3155', '0712.2716', '0903.4375', '1003.3590', '1202.1316', '0704.3084', '1001.3651', '1012.1201', '1108.2291', '0909.1739', '0806.4688', '1011.5154', '0803.1447', '0805.0332', '0807.4146', '1005.1132', '0902.1539', '0809.5268'] 

Scores :  [1.0, 0.15, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [ ]:
data_Nmost[0]['id'] # id of the first document (input)

'1002.3982'

We notice that the document has a 100% similarity with himself ! Let's have a look to the second most relevant document found

In [ ]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")

In [ ]:
Id_list = [doc["id"] for doc in idx_to_id_Nmost]
i = find_index(Id_list,Most_similar[1])
abstract = data_Nmost[i]["abstract"]
print(abstract)

we present a systematic group space scan of discrete abelian flavor
symmetries for lepton mass models that produce nearly tribimaximal lepton
mixing in our models small neutrino masses are generated by the typei seesaw
mechanism the lepton mass matrices emerge from higherdimension operators via
the froggattnielsen mechanism and are predicted as powers of a single
expansion parameter epsilon that is of the order of the cabibbo angle
thetacsimeq 02 we focus on solutions that can give close to tribimaximal
lepton mixing with a very small reactor angle theta13approx 0 and find
several thousand explicit such models that provide an excellent fit to current
neutrino data the models are rather general in the sense that large leptonic
mixings can come from the charged leptons andor neutrinos moreover in the
neutrino sector both left and righthanded neutrinos can mix maximally we
also find a new relation theta13lesssimepsilon3 for the reactor angle
and a new sum rule theta23approxpi4epsilonsqrt2

This document is about neutrinos

In [18]:
print(data_Nmost[0]["abstract"])

new particles at the tev scale can decay hadronically with strongly
collimated jets thus the standard reconstruction methods based on
invariantmasses of wellseparated jets can fail we discuss how to identify
such particles in pp collisions at the lhc using jet shapes which help to
reduce the contribution of qcdinduced events we focus on a rather generic
example x to ttbar to hadrons with x being a heavy particle but the approach
is well suited for reconstruction of other decay channels characterized by a
cascade decay of known states


The original document talks about particles : which is a similar subject !!